# カメラ修理データ 類似事例検索

対話的に各ステップを実行して結果を確認しながら進めるNotebook。

## 事前準備

```bash
pip install pyodbc pandas aiohttp requests tenacity nest_asyncio python-dotenv pyarrow
```

Macの場合はODBC Driverも必要:
```bash
brew install msodbcsql18
```

`.env` ファイルに認証情報を記載:
```
DIFY_API_KEY_SCHEMA=app-xxxxxxxxxxxx
DIFY_API_KEY_TAGGING=app-yyyyyyyyyyyy
MSSQL_SERVER=...
MSSQL_DATABASE=...
MSSQL_USER=...
MSSQL_PASSWORD=...
```

## 1. モジュール読み込み

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import json
from pathlib import Path

import config
import db
import loader
import preprocess
import dify_client
import scoring

## 2. 問い合わせ内容を定義

In [ ]:
inquiry_text = """
EOS R7で、冬の屋外撮影時にAFが迷う現象が複数のお客様から報告されている。
特定のレンズ装着時に発生するのか、個体差なのか、低温環境に起因するのか切り分けたい。
過去に同様の症状で修理されたデータがあれば参照したい。
"""

## 3. Dify 1回目：タグスキーマ生成

In [ ]:
schema = dify_client.generate_tag_schema(inquiry_text, max_detail_axes=4)

print("=== クエリ要約 ===")
print(schema["query_summary"])
print("\n=== 生成された軸 ===")
for ax in schema["axes"]:
    print(f"\n[{ax['tier']}] {ax['name']} (priority={ax.get('priority', '-')})")
    print(f"  {ax['description']}")
    print(f"  候補: {ax['candidates']}")

### スキーマを確認して、必要なら手動調整

動的生成なので、出力が意図と違えば問い合わせ文を修正して再実行するか、直接編集する。

In [ ]:
# 例：候補を手動で追加したい場合
# for ax in schema["axes"]:
#     if ax["name"] == "発生環境温度":
#         ax["candidates"] = ["低温（氷点下）", "低温（0-10℃）", "常温", "高温", "不明"]

## 4. データ取得 (SQL または CSV)

`loader` モジュール経由でデータを取得する。SQL と CSV のどちらでも
論理カラム名(`repair_id`, `user_comment`, ...)で揃ったDataFrameが返るので、
以降のセル(前処理・タグ付け)は読み込み元を意識せず同じ手順で実行できる。

### 利用可能なマッピングプリセット

`mappings/` ディレクトリの JSON ファイル(`sample_japan.json`, `sample_overseas.json` 等)。
新しいフォーマットは GUI もしくは `loader.save_mapping()` で追加できる。

In [ ]:
# === 利用可能なマッピングを確認 ===
for m in loader.list_mappings():
    print(f"  [{m['name']}] {m['display_name']}: {m['description']}")

In [ ]:
# === CSV から読み込み ===
# 実際のファイルパスとマッピング名を指定
csv_path = "data/repair_export.csv"  # ← 実際のパスに変更
mapping_name = "sample_japan"          # ← 該当するマッピング名に変更

repair_df = loader.load_from_csv(csv_path, mapping_name=mapping_name)
print(f"\n取得件数: {len(repair_df)}")
repair_df.head()

In [ ]:
# === SQL から読み込み(代替) ===
# CSV ではなく SQL を使う場合はこちらを実行。
# SQL の SELECT 句のカラム名が論理名と一致していればマッピング不要。
# 違う場合は mapping_name を指定する。

# sql = '''
#     SELECT TOP 500
#         repair_id, user_comment, repair_comment, internal_1, internal_2,
#         model, repair_date, country_code
#     FROM repair_records
#     WHERE model = ? AND repair_date >= ?
#     ORDER BY repair_date DESC
# '''
# repair_df = loader.load_from_sql(sql, params=("EOS R7", "2024-01-01"))
# print(f"取得件数: {len(repair_df)}")
# repair_df.head()

## 5. 前処理・整形

In [ ]:
records = preprocess.prepare_records(repair_df)
batches = preprocess.chunk_records(records, batch_size=config.BATCH_SIZE)

print(f"総レコード: {len(records)}")
print(f"バッチ数: {len(batches)}（1バッチ{config.BATCH_SIZE}件）")
print(f"\n=== 整形サンプル（1件目） ===")
print(records[0]["records"])
print(f"\nメタ情報: {records[0]['meta']}")

### 言語・コメント長の分布を確認

In [ ]:
meta_df = pd.DataFrame([r["meta"] for r in records])
print("=== 言語分布 ===")
print(meta_df["language"].value_counts())
print("\n=== コメント長tier分布 ===")
print(meta_df["length_tier"].value_counts())

## 6. Dify 2回目：タグ付け（非同期バッチ実行）

500件×10件/バッチ×並列5 だと、大体数分〜10分程度。

In [ ]:
from tqdm.notebook import tqdm

pbar = tqdm(total=len(batches), desc="タグ付け中")

def progress(done, total, last_result):
    pbar.update(1)
    if not last_result["success"]:
        pbar.write(f"⚠️ batch {last_result['batch_idx']}: {last_result['error'][:100]}")

batch_results = dify_client.run_tagging_sync(
    tag_schema=schema,
    inquiry_summary=schema["query_summary"],
    batches=batches,
    progress_callback=progress,
)
pbar.close()

success_count = sum(1 for b in batch_results if b["success"])
print(f"\n成功バッチ: {success_count}/{len(batches)}")

## 7. 結果をDataFrameに展開

In [ ]:
tagged_df = scoring.flatten_tagging_results(batch_results, schema)
print(f"タグ付け完了: {len(tagged_df)}件")
tagged_df.head()

### タグの分布を確認

In [ ]:
for ax in schema["axes"]:
    col = f"tag__{ax['name']}"
    if col in tagged_df.columns:
        print(f"\n=== {ax['name']} ({ax['tier']}) ===")
        print(tagged_df[col].value_counts(dropna=False))

## 8. 絞り込み：問い合わせタグを指定してスコアリング

問い合わせに対応するタグを指定。コア軸は必須、detail軸は該当するものだけ。

In [ ]:
# 問い合わせ側のタグを定義（スキーマを見ながら手動で指定）
query_tags = {
    "症状カテゴリ": "AF系",
    "AF症状の詳細": "AF迷い",
    "発生環境温度": "低温",
}

ranked_df = scoring.rank_results(
    tagged_df,
    query_tags=query_tags,
    schema=schema,
    min_relevance=0.3,
    top_n=50,
)

print(f"絞り込み結果: {len(ranked_df)}件")
ranked_df[[
    "repair_id", "match_score", "overall_relevance", 
    "relevance_reason", "language_detected"
]].head(20)

### 上位結果の根拠を詳しく確認

In [ ]:
for _, row in ranked_df.head(5).iterrows():
    print(f"\n{'='*60}")
    print(f"repair_id: {row['repair_id']} | score: {row['match_score']:.2f}")
    print(f"relevance: {row['overall_relevance']:.2f} | {row['relevance_reason']}")
    print()
    for ax in schema["axes"]:
        name = ax["name"]
        tag = row.get(f"tag__{name}")
        conf = row.get(f"conf__{name}", 0)
        ev = row.get(f"evidence__{name}", "")
        print(f"  [{name}] {tag} (conf={conf:.2f})")
        print(f"    根拠: {ev}")

## 9. 結果を保存（Parquet + CSV）

Tableauから直接開けるようCSVも出力。

In [ ]:
paths = scoring.save_results(
    tagged_df=tagged_df,
    ranked_df=ranked_df,
    schema=schema,
    inquiry_text=inquiry_text,
    tag="EOS_R7_AF_lowtemp",
)

for k, v in paths.items():
    print(f"{k}: {v}")

## 10. 再利用：失敗バッチだけ再実行する場合

In [ ]:
failed_batches = [b for b in batch_results if not b["success"]]
print(f"失敗バッチ: {len(failed_batches)}")

if failed_batches:
    # 失敗したバッチのrepair_idを集めて再バッチ化
    failed_ids = set()
    for b in failed_batches:
        failed_ids.update(b["input_ids"])
    
    retry_records = [r for r in records if r["repair_id"] in failed_ids]
    # 再実行時はバッチサイズを半分にすると破綻しにくい
    retry_batches = preprocess.chunk_records(retry_records, batch_size=5)
    
    retry_results = dify_client.run_tagging_sync(
        tag_schema=schema,
        inquiry_summary=schema["query_summary"],
        batches=retry_batches,
    )
    
    # 成功分を元の結果に追加してから再度flatten
    all_results = [b for b in batch_results if b["success"]] + [b for b in retry_results if b["success"]]
    tagged_df = scoring.flatten_tagging_results(all_results, schema)
    print(f"再実行後: {len(tagged_df)}件")